# Guardrails Agent — PII Detection & Output Validation
## Block the Bad, Pass the Good — PII Detection and Output Validation
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/61-guardrails-agent/guardrails_workbook.ipynb)

Production LLM applications need guardrails on both *input* and *output*. This workshop builds a **two-sided guardrail agent**: an `InputGuard` blocks queries that contain PII (SSNs, email addresses, credit card numbers) or are off-topic, while an `OutputGuard` enforces a word-count cap before the answer reaches the user. Both guards are Pydantic models validated via `with_structured_output()` — no regex hand-rolling. By the end you will understand *how* to build reliable dual-sided guards and *why* structured output is more robust than free-text parsing.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why dual-sided guards matter |
| 2 | **PII Detection** — Regex patterns + `InputGuard` Pydantic model |
| 3 | **Output Validation** — `OutputGuard` word-count enforcement |
| 4 | **Conditional Routing** — Block or pass based on guard decision |
| 5 | **Full Pipeline** — validate_input → agent → validate_output |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Why Dual-Sided Guards Matter

```
Query → [InputGuard] → blocked? ─YES─► "Blocked" response
                     ↓ NO
                   [Agent]
                     ↓
                [OutputGuard] → fails? ─YES─► truncate / flag
                     ↓ passes
                   Answer
```

| Guard | What it catches | How |
|---|---|---|
| `InputGuard` | PII, off-topic queries | LLM structured output + regex pre-check |
| `OutputGuard` | Verbosity, hallucination markers, policy violations | Word count, keyword scan |

**Key insight:** Structured output (`with_structured_output`) forces the LLM into a Pydantic model — no brittle parsing, no missing fields.

In [ ]:
import re
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

MAX_ANSWER_WORDS = 150

# Regex first: cheaper and instant; LLM classifier catches semantic PII that regex misses
PII_PATTERNS = [
    r"\b\d{3}-\d{2}-\d{4}\b",           # SSN
    r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # email
    r"\b\d{16}\b",                        # credit card
]

# temperature=0 → deterministic outputs; raise to 0.7+ for varied creative responses
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
print(f"Model ready. MAX_ANSWER_WORDS={MAX_ANSWER_WORDS}")

---
## Part 2 — PII Detection with `InputGuard`

In [ ]:
class InputGuard(BaseModel):
    is_on_topic: bool
    contains_pii: bool
    reason: str

def has_pii(text: str) -> bool:
    return any(re.search(p, text) for p in PII_PATTERNS)

# with_structured_output forces the model to return a validated Pydantic object — no manual JSON parsing
input_checker = llm.with_structured_output(InputGuard)

# Test queries
TEST_QUERIES = [
    "What is the refund policy?",
    "My SSN is 123-45-6789, can you help me?",
    "Tell me a joke about penguins.",
    "Contact me at user@example.com to discuss my account.",
]

print("=== InputGuard results ===")
for q in TEST_QUERIES:
    regex_pii = has_pii(q)
    result = input_checker.invoke([
        {"role": "system", "content": "You are a content moderator. Check if the query is on-topic (customer support) and contains PII."},
        {"role": "user", "content": q}
    ])
# OR-merge: regex catches obvious patterns fast; LLM catches semantic variants ('my identifier is ...')
    final_pii = regex_pii or result.contains_pii
    status = "BLOCKED" if (not result.is_on_topic or final_pii) else "PASS"
    print(f"[{status}] {q[:50]}")
    print(f"  on_topic={result.is_on_topic}, pii={final_pii} (regex={regex_pii}), reason={result.reason}")

---
## Part 3 — Output Validation with `OutputGuard`

In [ ]:
class OutputGuard(BaseModel):
    passes: bool
    word_count: int
    reason: str

# Same structured-output trick for the output side — model returns word_count + passes flag in one call
output_checker = llm.with_structured_output(OutputGuard)

SAMPLE_ANSWERS = [
    "Our refund policy is 30 days with a receipt.",
    " ".join(["word"] * 200),  # too long
]

print("=== OutputGuard results ===")
for ans in SAMPLE_ANSWERS:
    result = output_checker.invoke([
        {"role": "system", "content": f"Check if this answer passes policy: under {MAX_ANSWER_WORDS} words, no harmful content."},
        {"role": "user", "content": ans[:500]}
    ])
    print(f"passes={result.passes}, words={result.word_count}, reason={result.reason}")
    print(f"  Preview: {ans[:60]}...")

---
## Part 4 — Conditional Routing

After `validate_input`, a conditional edge routes to either:
- `"blocked"` → if `state["blocked"]` is True (PII found or off-topic)
- `"agent"` → if input passes all checks

```python
def route_input(state):
    return "blocked" if state["blocked"] else "agent"
```

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Demonstrate routing logic manually
def check_input(query: str):
    regex_pii = has_pii(query)
    guard = input_checker.invoke([
        SystemMessage(content="Check if this customer support query is on-topic and contains PII."),
        HumanMessage(content=query)
    ])
    blocked = not guard.is_on_topic or guard.contains_pii or regex_pii
    block_reason = guard.reason if blocked else ""
    return {"blocked": blocked, "block_reason": block_reason, "guard": guard}

for q in TEST_QUERIES:
    r = check_input(q)
    route = "blocked" if r["blocked"] else "agent"
    print(f"→ {route:7s}  |  {q[:55]}")

---
## Part 5 — Full LangGraph Pipeline

```
START → validate_input → [blocked → END]
                       → [agent → validate_output → END]
```

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

# TypedDict gives LangGraph typed state keys at zero runtime cost — no __init__, no dataclass overhead
class GuardrailsState(TypedDict):
    query: str; input_decision: dict; answer: str
    output_decision: dict; blocked: bool; block_reason: str

def validate_input_node(state):
    regex_pii = has_pii(state["query"])
    g = input_checker.invoke([
        SystemMessage(content="Check if this customer support query is on-topic and contains PII."),
        HumanMessage(content=state["query"])
    ])
    blocked = not g.is_on_topic or g.contains_pii or regex_pii
    return {"input_decision": g.model_dump(), "blocked": blocked,
            "block_reason": g.reason if blocked else ""}

def agent_node(state):
    ans = llm.invoke([
        SystemMessage(content=f"You are a helpful customer support agent. Answer in under {MAX_ANSWER_WORDS} words."),
        HumanMessage(content=state["query"])
    ]).content.strip()
    return {"answer": ans}

def blocked_node(state):
    return {"answer": f"[BLOCKED] {state['block_reason']}"}

def validate_output_node(state):
    g = output_checker.invoke([
        SystemMessage(content=f"Check if this answer passes policy: under {MAX_ANSWER_WORDS} words, no harmful content."),
        HumanMessage(content=state["answer"])
    ])
    answer = state["answer"]
    if not g.passes:
        words = answer.split()
        answer = " ".join(words[:MAX_ANSWER_WORDS]) + " [truncated]"
    return {"output_decision": g.model_dump(), "answer": answer}

# route_input return value ('blocked'/'agent') must match the keys in add_conditional_edges dict exactly
def route_input(state):
    return "blocked" if state["blocked"] else "agent"

g = StateGraph(GuardrailsState)
for name, fn in [("validate_input", validate_input_node), ("agent", agent_node),
                 ("blocked", blocked_node), ("validate_output", validate_output_node)]:
    g.add_node(name, fn)
g.add_edge(START, "validate_input")
# add_conditional_edges: after validate_input, call route_input() and route to the matching node key
g.add_conditional_edges("validate_input", route_input, {"blocked": "blocked", "agent": "agent"})
g.add_edge("blocked", END)
g.add_edge("agent", "validate_output")
g.add_edge("validate_output", END)
# compile() locks the graph topology; after this point the structure is immutable
app = g.compile()

DEMO_QUERIES = [
    "What is your return policy?",
    "My credit card number is 1234567890123456 and I want a refund.",
]

for q in DEMO_QUERIES:
    init = {"query": q, "input_decision": {}, "answer": "",
            "output_decision": {}, "blocked": False, "block_reason": ""}
    r = app.invoke(init)
    print(f"Q: {q}")
    print(f"Blocked: {r['blocked']}  |  Answer: {r['answer'][:120]}")
    print()

---
### Exercise 1 — Add a Phone Number Pattern
Add a regex for US phone numbers (e.g., `r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"`) to `PII_PATTERNS` and test with `"Call me at 555-867-5309"`. Does it get blocked?

### Exercise 2 — Strict Output Guard
Change `MAX_ANSWER_WORDS` to `50` and rerun the pipeline. How does the `validate_output_node` truncation change the answer? What are the trade-offs of very strict output limits?

In [ ]:
# Exercise 1 — phone number PII
PII_PATTERNS_EX = PII_PATTERNS + [r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"]

phone_query = "Call me at 555-867-5309 for support."
print(f"Phone PII detected: {any(re.search(p, phone_query) for p in PII_PATTERNS_EX)}")

# Exercise 2 — strict 50-word limit
MAX_ANSWER_WORDS_EX = 50
long_answer = "Our return policy allows customers to return any item within 30 days of purchase " * 5
words = long_answer.split()
if len(words) > MAX_ANSWER_WORDS_EX:
    truncated = " ".join(words[:MAX_ANSWER_WORDS_EX]) + " [truncated]"
    print(f"Original: {len(words)} words → Truncated: {len(truncated.split())} words")
    print(truncated)

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*